In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyClientCredentials
import time
import requests

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
load_dotenv()
client_id = os.getenv('SPOTIPY_CLIENT_ID')
client_secret = os.getenv('SPOTIPY_CLIENT_SECRET')
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=client_id, 
    client_secret=client_secret
))

In [ ]:
def music_data_reader(file):
    music_data = pd.read_csv(file)
    return music_data

In [ ]:
music_data = music_data_reader('C:/Users/maxph/OneDrive/Desktop/CLARIFY/ExpandedMusicData.csv')
music_data

In [ ]:
def fetch_spotify_id(row):
    # Convert to string in case of missing data (NaN)
    song = str(row['Song']) 
    artist = str(row['Artist'])
    
    # Format the query specifically for the Spotify search engine
    query = f"track:{song} artist:{artist}"
    
    try:
        # Search for 1 track matching the query
        results = sp.search(q=query, type='track', limit=1)
        items = results['tracks']['items']
        
        # If the list is not empty, extract the ID
        if items:
            return items[0]['id']
        else:
            print(f"No match found for: {song} by {artist}")
            return None
            
    except Exception as e:
        print(f"API Error fetching {song}: {e}")
        return None

In [ ]:
music_data['SONG_ID'] = music_data.apply(fetch_spotify_id, axis=1)
music_data

In [ ]:
# Ensure your RapidAPI key is loaded
load_dotenv()
RAPIDAPI_KEY = os.getenv('RAPIDAPI_KEY')

def fetch_all_soundnet_features(row):
    # Grab the Spotify ID we generated earlier
    spotify_id = row.get('SONG_ID')
    
    # Define the exact columns we expect to create
    feature_columns = [
        'SoundNet_ID', 'Key', 'Mode', 'Camelot', 
        'Tempo', 'Duration', 'Popularity', 'Energy', 'Danceability', 
        'Happiness', 'Acousticness', 'Instrumentalness', 'Liveness', 
        'Speechiness', 'Loudness'
    ]
    
    # If there is no ID, return a row of empty values
    if pd.isna(spotify_id) or not spotify_id:
        return pd.Series({col: None for col in feature_columns})

    # The API endpoint
    url = f"https://track-analysis.p.rapidapi.com/pktx/spotify/{spotify_id}"
    
    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": "track-analysis.p.rapidapi.com"
    }
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            
            # Rate limiting sleep (1 second between successful calls)
            time.sleep(1)
            
            # Map the JSON data directly into a pandas Series
            return pd.Series({
                'SoundNet_ID': data.get('id'),
                'Key': data.get('key'),
                'Mode': data.get('mode'),
                'Camelot': data.get('camelot'),
                'Tempo': data.get('tempo'),
                'Duration': data.get('duration'),
                'Popularity': data.get('popularity'),
                'Energy': data.get('energy'),
                'Danceability': data.get('danceability'),
                'Happiness': data.get('happiness'),
                'Acousticness': data.get('acousticness'),
                'Instrumentalness': data.get('instrumentalness'),
                'Liveness': data.get('liveness'),
                'Speechiness': data.get('speechiness'),
                'Loudness': data.get('loudness')
            })
        else:
            print(f"API Error {response.status_code} for ID {spotify_id}: {response.text}")
            
    except Exception as e:
        print(f"Network error for ID {spotify_id}: {e}")
    
    # Fallback rate limiting in case of errors
    time.sleep(1) 
    
    # Return empty data if the API call failed
    return pd.Series({col: None for col in feature_columns})

In [ ]:
print("Fetching comprehensive audio features from SoundNet...")
new_audio_features = music_data.apply(fetch_all_soundnet_features, axis=1)
print("Merging new columns into the dataset...")
music_data = pd.concat([music_data, new_audio_features], axis=1)
music_data

In [ ]:
music_data.to_csv('C:/Users/maxph/OneDrive/Desktop/CLARIFY/audio_features.csv', index=False)